# EDR-REDNet Ablation Study -- Variant D (Full Model)
**Variant D = Full EDR-REDNet: EdgeBlock + Sobel Input + Combined Loss Function**

| Variant | EdgeBlock | Sobel Input | Sobel Loss (a=0.3) | VGG Loss (b=0.05) | HU Loss (g=0.01) |
|---------|-----------|-------------|-------------------|------------------|------------------|
| A -- RED-CNN Baseline | No  | No  | No  | No  | No  |
| B -- + EdgeBlock      | Yes | No  | No  | No  | No  |
| C -- + Sobel Input    | Yes | Yes | No  | No  | No  |
| **D -- Full (this)**  | Yes | Yes | Yes | Yes | Yes |

**Tat ca Variant deu dung: max_iterations=31000, mbs=16, lr=9.583e-05**

> Chay lan luot tu Cell 1 den Cell 6.

In [1]:
# Cell 1: Setup
import os
os.environ["WANDB_MODE"] = "offline"
os.environ["WANDB_START_METHOD"] = "thread"

!git clone https://github.com/minhvuongle2004/lung-diagnosis.git
%cd /kaggle/working/lung-diagnosis/ldct-benchmark
!pip install -e . -q

print("Setup done!")

Cloning into 'lung-diagnosis'...
remote: Enumerating objects: 516, done.
remote: Counting objects: 100% (516/516), done.
remote: Compressing objects: 100% (411/411), done.
remote: Total 516 (delta 159), reused 447 (delta 90), pack-reused 0 (from 0)
Receiving objects: 100% (516/516), 9.11 MiB | 29.42 MiB/s, done.
Resolving deltas: 100% (159/159), done.
/kaggle/working/lung-diagnosis/ldct-benchmark
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for ldct-benchmark (pyproject.toml) ... done
Setup done!


In [2]:
# Cell 2: Tim duong dan du lieu
import os

data_path = None
for dataset_slug in os.listdir("/kaggle/input"):
    base = f"/kaggle/input/{dataset_slug}"
    for root, dirs, files in os.walk(base):
        if "LDCT-and-Projection-data" in dirs:
            data_path = root
            sub = os.path.join(root, "LDCT-and-Projection-data")
            patients = sorted(os.listdir(sub))
            print(f"Datafolder: {data_path}")
            print(f"So benh nhan: {len(patients)}")
            break
    if data_path:
        break

if data_path is None:
    print("KHONG tim thay LDCT-and-Projection-data!")

Datafolder: /kaggle/input/datasets/vuongleminh604/ldct-data/data
So benh nhan: 104


In [3]:
# Cell 3: Loc info.yml
import os, yaml, shutil

info_path = "ldctbench/data/info.yml"
with open(info_path) as f:
    info = yaml.safe_load(f)

ldct_dir = os.path.join(data_path, "LDCT-and-Projection-data")
available = set(os.listdir(ldct_dir))

new_info = {k: v for k, v in info.items() if k not in ["train_set", "val_set", "test_set"]}
for split in ["train_set", "val_set", "test_set"]:
    original = info.get(split, [])
    filtered = []
    for entry in original:
        pid = entry["id"]
        if pid not in available:
            continue
        input_rel = entry["input"].replace("./LDCT-and-Projection-data/", "")
        input_full = os.path.join(ldct_dir, input_rel)
        if not os.path.exists(input_full):
            continue
        actual_files = sorted([f for f in os.listdir(input_full) if f.endswith(".dcm")])
        if len(actual_files) == 0:
            continue
        entry = dict(entry)
        entry["n_slices"] = len(actual_files)
        filtered.append(entry)
    new_info[split] = filtered
    print(f"{split}: {len(original)} -> {len(filtered)} benh nhan")

shutil.copy(info_path, info_path + ".bak")
with open(info_path, "w") as f:
    yaml.dump(new_info, f, default_flow_style=False, allow_unicode=True)
print("Da luu info.yml moi")

train_set: 34 -> 34 benh nhan
val_set: 4 -> 4 benh nhan
test_set: 9 -> 9 benh nhan
Da luu info.yml moi


In [4]:
# Cell 4: Patch dinh dang file .dcm (neu can)
import os, re

ldct_dir = os.path.join(data_path, "LDCT-and-Projection-data")
sample_dcm = None

for patient in sorted(os.listdir(ldct_dir)):
    patient_path = os.path.join(ldct_dir, patient)
    for root, dirs, files in os.walk(patient_path):
        dcm_files = sorted([f for f in files if f.endswith(".dcm")])
        if dcm_files:
            sample_dcm = dcm_files[0]
            break
    if sample_dcm:
        break

print(f"Sample DCM: {sample_dcm}")

ldct_mayo_path = "ldctbench/data/LDCTMayo.py"
with open(ldct_mayo_path, "r") as f:
    content = f.read()

if sample_dcm and not sample_dcm.startswith("0"):
    match = re.match(r'^(.*?)(\d+)\.dcm$', sample_dcm)
    if match:
        prefix = match.group(1)
        digits = len(match.group(2))
        old_line = '        return "{}.dcm".format(str(idx).zfill(8))'
        new_line = f'        return "{prefix}{{}}.dcm".format(str(idx).zfill({digits}))'
        if old_line in content:
            content = content.replace(old_line, new_line)
            with open(ldct_mayo_path, "w") as f:
                f.write(content)
            print(f"Patched LDCTMayo.py -> format '{prefix}{{:0{digits}d}}.dcm'")
        else:
            print("Format cu khong tim thay de patch.")
else:
    print("Format da dung (00000XXX.dcm), khong can patch")

Sample DCM: 00000001.dcm
Format da dung (00000XXX.dcm), khong can patch


In [5]:
# Cell 5: Train VARIANT D — Full EDR-REDNet (Best Performance Config)
# ─────────────────────────────────────────────────────────────────────
# Config giong het seed2024 (da proven):                               
#   loss_alpha=0.3  → Sobel Edge Loss (manh hon C 3x)                 
#   loss_beta=0.05  → Perceptual VGG Loss                             
#   loss_gamma=0.01 → HU Loss                                         
#   max_iterations=45000 (~7h tren T4)                              
# ─────────────────────────────────────────────────────────────────────
import glob, os

assert data_path is not None, "Chay Cell 2 truoc!"

VARIANT = "D"
seed = 1339
max_iterations = 31000

resume_config = ""
checkpoints = glob.glob(f"/kaggle/input/**/variantD_seed{seed}_best_*.pt", recursive=True)
if checkpoints:
    ckpt_path = checkpoints[0]
    resume_config = f"resume: '{ckpt_path}'"
    print(f"Resume tu: {ckpt_path}")
else:
    print(f"Train Variant D tu dau (~7h tren T4)")

config = f"""trainer: edrrednet
seed: {seed}
datafolder: {data_path}
{resume_config}
optimizer: adam
lr: 9.583e-05
adam_b1: 0.9
adam_b2: 0.999
loss_alpha: 0.3
loss_beta: 0.05
loss_gamma: 0.01
num_edge_blocks: 2
use_sobel_input: true
mbs: 16
max_iterations: {max_iterations}
data_subset: 1.0
patchsize: 128
iterations_before_val: 500
valsamples: 8
data_norm: meanstd
num_workers: 2
cuda: true
devices: 0
"""

config_path = f"configs/ablation_variantD_seed{seed}.yaml"
with open(config_path, "w", encoding="utf-8") as f:
    f.write(config)

print(f"Config saved: {config_path}")
print("Bat dau training Variant D...")
!python -u -m ldctbench.scripts.train --config {config_path}

Train Variant D tu dau (~7h tren T4)
Config saved: configs/ablation_variantD_seed1339.yaml
Bat dau training Variant D...
wandb: WARNING `start_method` is deprecated and will be removed in a future version of wandb. This setting is currently non-functional and safely ignored.
wandb: Tracking run with wandb version 0.25.0
wandb: W&B syncing is set to `offline` in this directory. Run `wandb online` or set WANDB_MODE=online to enable cloud syncing.
wandb: Run data is saved locally in /kaggle/working/lung-diagnosis/ldct-benchmark/wandb/offline-run-20260520_024408-qaxq5nyk
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. 

In [6]:
# Cell 6: Luu checkpoint ra Output
# Chay sau khi training xong, truoc khi bam Save Version
import glob, shutil, os

output_dir = "/kaggle/working"
VARIANT = "D"
seed = 1339

checkpoints = glob.glob("wandb/offline-run-*/files/best_*.pt")
print(f"Checkpoints found: {checkpoints}")
for ckpt in checkpoints:
    dest = os.path.join(output_dir, f"variant{VARIANT}_seed{seed}_{os.path.basename(ckpt)}")
    shutil.copy(ckpt, dest)
    print(f"Saved: {dest}")

csv_files = glob.glob("wandb/offline-run-*/files/*.csv")
for csv in csv_files:
    dest = os.path.join(output_dir, f"variant{VARIANT}_seed{seed}_{os.path.basename(csv)}")
    shutil.copy(csv, dest)
    print(f"Log saved: {dest}")

print("\nDone! Nho bam 'Save Version' de luu output.")

Checkpoints found: ['wandb/offline-run-20260520_024408-qaxq5nyk/files/best_SSIM.pt']
Saved: /kaggle/working/variantD_seed1339_best_SSIM.pt
Log saved: /kaggle/working/variantD_seed1339_Losses.csv
Log saved: /kaggle/working/variantD_seed1339_Metrics.csv

Done! Nho bam 'Save Version' de luu output.
